In [19]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [20]:
load_dotenv()
model=ChatOpenAI()

In [21]:
class PhraseState(TypedDict):
    topic: str
    phrase: str
    example: str




In [22]:
def gen_phrase(state: PhraseState):
    prompt=f"Generate a phrase related to given topic: {state['topic']}"
    phrase= model.invoke(prompt).content
    return {"phrase": phrase}

In [23]:
def phrase_eg(state: PhraseState):
    prompt=f'Generate 3 examples for the phrase: {state["phrase"]}'
    examples = model.invoke(prompt).content
    return {"example": examples}

In [24]:
graph=StateGraph(PhraseState)

graph.add_node('gen_phrase', gen_phrase)
graph.add_node('phrase_eg', phrase_eg)

graph.add_edge(START, 'gen_phrase')
graph.add_edge('gen_phrase','phrase_eg')
graph.add_edge('phrase_eg', END)

Checkpointer=InMemorySaver()
workflow=graph.compile(checkpointer=Checkpointer)

In [25]:
# workflow

In [26]:
config={"configurable": {"thread_id":"1"}}
workflow.invoke({"topic": "daily life"}, config=config)

{'topic': 'daily life',
 'phrase': '"Embrace the beauty of the mundane in your daily life."',
 'example': '1. Take a moment to appreciate the gentle rhythm of your morning routine - the sound of the coffee brewing, the warmth of the sun streaming through the window, and the peaceful quiet before the day begins.\n\n2. Notice the small details in your surroundings as you go about your day - the way the light dances through the leaves of a tree, the laughter of children playing in the park, or the sweet smell of freshly cut grass.\n\n3. Find joy in the simplicity of everyday tasks - the satisfaction of a clean kitchen, the comfort of a home-cooked meal, or the warmth of a cozy blanket on a chilly evening.'}

In [27]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'daily life', 'phrase': '"Embrace the beauty of the mundane in your daily life."', 'example': '1. Take a moment to appreciate the gentle rhythm of your morning routine - the sound of the coffee brewing, the warmth of the sun streaming through the window, and the peaceful quiet before the day begins.\n\n2. Notice the small details in your surroundings as you go about your day - the way the light dances through the leaves of a tree, the laughter of children playing in the park, or the sweet smell of freshly cut grass.\n\n3. Find joy in the simplicity of everyday tasks - the satisfaction of a clean kitchen, the comfort of a home-cooked meal, or the warmth of a cozy blanket on a chilly evening.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13e76c-d54d-6845-8002-1bb9d35e7f14'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-22T18:12:19.714874+00:00', parent_config={'configurable': {

In [28]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'daily life', 'phrase': '"Embrace the beauty of the mundane in your daily life."', 'example': '1. Take a moment to appreciate the gentle rhythm of your morning routine - the sound of the coffee brewing, the warmth of the sun streaming through the window, and the peaceful quiet before the day begins.\n\n2. Notice the small details in your surroundings as you go about your day - the way the light dances through the leaves of a tree, the laughter of children playing in the park, or the sweet smell of freshly cut grass.\n\n3. Find joy in the simplicity of everyday tasks - the satisfaction of a clean kitchen, the comfort of a home-cooked meal, or the warmth of a cozy blanket on a chilly evening.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13e76c-d54d-6845-8002-1bb9d35e7f14'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-22T18:12:19.714874+00:00', parent_config={'configurable': 

In [29]:
config2={"configurable": {"thread_id":"2"}}
workflow.invoke({"topic": "Bussiness"}, config=config2)

{'topic': 'Bussiness',
 'phrase': '"Success in business requires strategic decisions and unwavering determination."',
 'example': '1. When facing tough competition in the market, a successful business owner made strategic decisions to cut costs and improve efficiency, all while maintaining unwavering determination to achieve long-term success.\n\n2. A startup company achieved great success by making strategic decisions to pivot their business model based on market feedback, all while showing unwavering determination to persevere through challenges and setbacks.\n\n3. In a highly competitive industry, a successful entrepreneur credited their strategic decisions to expand into new markets and invest in innovative technology, along with unwavering determination to overcome obstacles and achieve their business goals.'}

The state history of each thread could see using ".get_state_history"

In [31]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Bussiness', 'phrase': '"Success in business requires strategic decisions and unwavering determination."', 'example': '1. When facing tough competition in the market, a successful business owner made strategic decisions to cut costs and improve efficiency, all while maintaining unwavering determination to achieve long-term success.\n\n2. A startup company achieved great success by making strategic decisions to pivot their business model based on market feedback, all while showing unwavering determination to persevere through challenges and setbacks.\n\n3. In a highly competitive industry, a successful entrepreneur credited their strategic decisions to expand into new markets and invest in innovative technology, along with unwavering determination to overcome obstacles and achieve their business goals.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f13e777-5d12-64d9-8002-fbc5c604bb60'}}, metadata={'source': 

Time Travel: You can continue from any state using checkPoint id

In [32]:
workflow.get_state({"configurable":{"thread_id":"1","check_point_id":"1f13e777-460d-65da-8001-eee7b792e5dc"}})

StateSnapshot(values={'topic': 'daily life', 'phrase': '"Embrace the beauty of the mundane in your daily life."', 'example': '1. Take a moment to appreciate the gentle rhythm of your morning routine - the sound of the coffee brewing, the warmth of the sun streaming through the window, and the peaceful quiet before the day begins.\n\n2. Notice the small details in your surroundings as you go about your day - the way the light dances through the leaves of a tree, the laughter of children playing in the park, or the sweet smell of freshly cut grass.\n\n3. Find joy in the simplicity of everyday tasks - the satisfaction of a clean kitchen, the comfort of a home-cooked meal, or the warmth of a cozy blanket on a chilly evening.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f13e76c-d54d-6845-8002-1bb9d35e7f14'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-04-22T18:12:19.714874+00:00', parent_config={'configurable': {